<a href="https://colab.research.google.com/github/jaloaiza/genaiassignments/blob/main/assignments/Final%20Project/FinalProjDatasetGen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate Synthetic Training Data

<a target="_blank" href="https://colab.research.google.com/github/simonguest/CS-394/blob/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://github.com/simonguest/CS-394/raw/refs/heads/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://img.shields.io/badge/Download_.ipynb-blue" alt="Download .ipynb"/>
</a>

## Data generation settings

In [ ]:
NUM_TRAIN_EXAMPLES = 500  # @param {type:"number"}
NUM_VAL_EXAMPLES = 50  # @param {type:"number"}
NUM_TEST_EXAMPLES = 10 # @param {type:"number"}
TEMPERATURE = 0.8  # @param {type:"number"}

DATA_FOLDER = "./data/generated"
!mkdir -p {DATA_FOLDER}

DATAGEN_URL = "https://openrouter.ai/api/v1"
DATAGEN_MODEL = "openai/gpt-5-nano"

# Create it if it doesn't exist
#os.makedirs(DATA_FOLDER, exist_ok=True)

# Check if it exists
#print("Folder exists?", os.path.exists(DATA_FOLDER))
#print("Full path:", os.path.abspath(DATA_FOLDER))

## Dataset diversity

In [ ]:
COMMUNICATION_STYLES = [
    "casual_slang_heavy",
    "casual_clean",
    "formal_polite",
    "very_formal",
    "dry_minimal",
    "expressive_emotional",
    "sarcastic_ironic",
    "confused_uncertain",
    "curious_questioning",
    "assertive_direct"
]

COMMUNICATION_STYLES = [
    "casual_slang",
    "neutral",
    "formal",
    "very_short",
    "very_expressive",
    "sarcastic",
    "confused",
    "curious",
    "impatient",
    "friendly"
]

PLAYER_TYPES = [
    "explorer",
    "competitor",
    "casual_player",
    "roleplayer",
    "min_maxer",
    "social_player",
    "troll_light",
    "helper"
]

WRITING_PATTERNS = [
    "clean",
    "no_caps",
    "minor_typos",
    "heavy_typos",
    "shortened_words",
    "messy_spacing",
    "run_on"
]

VERBOSITY = [
    "one_word",
    "short_one_sentence",
    "short_two_sentences"
]

COMMUNICATION_STYLE_WEIGHTS = [0.3, 0.2, 0.1, 0.1, 0.1, 0.05, 0.05, 0.05, 0.025, 0.025]

PLAYER_TYPE_WEIGHTS = [0.2, 0.15, 0.25, 0.1, 0.1, 0.1, 0.05, 0.05]

WRITING_PATTERN_WEIGHTS = [0.3, 0.2, 0.2, 0.1, 0.1, 0.05, 0.05]

VERBOSITY_WEIGHTS = [0.2, 0.5, 0.3]


## Model for structured output

In [ ]:
from pydantic import BaseModel

class ProfilerExample(BaseModel):
    message: str
    communication_style: str
    player_type: str
    writing_pattern: str
    verbosity: str

## Get OpenRouter API key

In [ ]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
  load_dotenv()

## Conversation generation functions

In [ ]:
from httpx._transports.default import A
import openai
import os
from dataclasses import dataclass

client = openai.OpenAI(
    base_url=DATAGEN_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

def generate_completion(prompt: str) -> ProfilerExample | None:
    response = client.responses.parse(
        model=DATAGEN_MODEL,
        input=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        stream=False,
        text_format=ProfilerExample
    )

    return response.output_parsed

def create_conversation(
    communication_style: str,
    player_type: str,
    writing_pattern: str,
    verbosity: str
) -> ProfilerExample | None:

    prompt = f"""
    You are generating realistic user chat messages for a chatbot.
    The user will be talking about what they like in video games.

    Generate a single message that fits this profile:

    - Communication style: {communication_style}
    - Player type: {player_type}
    - Writing pattern: {writing_pattern}
    - Verbosity: {verbosity}

    IMPORTANT:
    - Do NOT mention the labels
    - The message should feel like a real player chatting
    - Match the writing pattern (typos, caps, etc.)
    - Match the verbosity strictly
    - Do not using emojis or special characters.

    Return structured output with:
    - message
    - communication_style
    - player_type
    - writing_pattern
    - verbosity
    """

    return generate_completion(prompt)

## Dataset generation functions

In [ ]:
import random
import json
from tqdm import tqdm

def generate_dataset(num_examples: int, filename: str) -> None:
    with open(filename, "w", encoding="utf-8") as f:
        for idx in tqdm(range(num_examples)):

          communication_style = random.choice(COMMUNICATION_STYLES)
          player_type = random.choice(PLAYER_TYPES)
          writing_pattern = random.choice(WRITING_PATTERNS)
          verbosity = random.choices(VERBOSITY, weights=VERBOSITY_WEIGHTS)[0]


          # Generate riddle
          conversation = None
          while conversation is None:
              conversation = create_conversation(
                  communication_style=communication_style,
                  player_type=player_type,
                  writing_pattern=writing_pattern,
                  verbosity=verbosity
                )


          # Build structured training template
          template = {
              "messages": [
                  {
                      "role": "user",
                      "content": conversation.message
                  }
              ],
              "labels": {
                  "communication_style": conversation.communication_style,
                  "player_type": conversation.player_type,
                  "writing_pattern": conversation.writing_pattern,
                  "verbosity": conversation.verbosity
              }
          }

          line = json.dumps(template, ensure_ascii=False) + "\n"
          f.write(line)
          f.flush()

        f.flush()
        f.close()

## Generate all the data!

In [ ]:
from datetime import datetime

TRAIN_FILE = f"{DATA_FOLDER}/train_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
VALID_FILE = f"{DATA_FOLDER}/valid_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
TEST_FILE = f"{DATA_FOLDER}/test_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"

#generate_dataset(NUM_TRAIN_EXAMPLES, TRAIN_FILE)
generate_dataset(NUM_VAL_EXAMPLES, VALID_FILE)
generate_dataset(NUM_TEST_EXAMPLES, TEST_FILE)


100%|██████████| 10/10 [03:58<00:00, 23.88s/it]


# Explanation

## What did I choose?

I decided to go with my own approach, of generating riddles. Somewhat along the lines of Creative Writing Helper, however I did not do a continuous narrative.

## Why did I choose this?
Starting with last week, I want to start building a practice project that will be similar to my final project. With my final project involving being puzzles/riddles. The last assignment taught me a lot about small models not being large enough to generate a multitude of riddles. Whether it having trouble with the complexity of a riddle, different topics of riddles, not knowing how to analyze a response.

## Shaping my Dataset

In my first stab at this project, I way overcomplicated the diversity dimensions. With each dimension I added, it added another layer of complexity and something to keep in mind with the prompt engineering. While a second pass at it did work, just didn't meet my expectations.

Additionally, I realized that the riddles that were being generated were just not fun and silly. They were way too "serious".

With both of those in mind my final iteration was the following dimensions:
1.   Riddle Topics
2.   Difficulty Levels (with weights)
3.   Riddle Length (with weights)
4.   Correctness (with weights)

**Riddle Topics:**
The riddle topics is quite long, however there are 3 "sections" to it. Each section targeting an average age period for a human (child, teen, adult). Looking forward towards my final project, I want the model to be able to generate different riddles targeted towards different audiences when prompted.

**Difficulty Levels:**
I wanted different difficulty levels for the riddles. Just so not all of them comically easy. However, I didn't want an even spread of difficult ones either, so I used weights to favor more of the easier - medium difficulty riddles.

**Riddle Length:**
Very similar to the difficulty levels, however, I mainly just wanted more short riddles. When doing all this generation of the data, I realized the more longer riddles tend to get unrealistic and more unpredictable. So the dataset is heavily favored towards more short riddles.

**Correctness:**
I am thankful to your help for refactoring this for me. During last week's assignment, the small model didn't actually know if the user's answer was correct or incorrect. So adding this in will definitely help.



## Observations

I kind of touched on a bit here and there, up above. Throughout multiple stages of generating small datasets, there were a lot of complications with the formatting, and the riddles themselves. I changed my prompts around a little bit the riddles themselves did get better. However, it wasn't until your help with simplifying the prompts that it got so much better. It fixed the formatting and got it all consistent.

Additionally, with me changing the topics and favoring more short riddles, it definitely cut out a lot of the "craziness" in the riddles.